# Phase 0b — vLLM Hook Spike

**Goal:** verify PyTorch forward hooks installed inside a vLLM Worker via the `WorkerExtension` API fire reliably when `llm.generate()` is called, **with vLLM's full V1 engine running** (`torch.compile` + CUDA graphs on, no `enforce_eager`).

**Go/no-go gate** for the TransformerLens vLLM source plan: if hooks miss under compiled graphs, the architecture is unworkable on current vLLM and we wait for [vLLM RFC #36998](https://github.com/vllm-project/vllm/issues/36998).

## How to run

1. **Runtime → Change runtime type → T4 GPU** (or better). CPU/TPU runtimes will fail.
2. Add an `HF_TOKEN` secret in Colab (🔑 sidebar) if the chosen model is gated. Default model (`Qwen2.5-0.5B-Instruct`) is **not** gated; you can skip auth unless you swap to Llama-3.2-1B.
3. **Runtime → Run all.** ~5–10 minutes on T4 (most of it is model download + vLLM compilation).
4. Paste the outputs of the **Phase 0b Summary** cell back to Claude.

## Pass criteria (all must hold)

1. `LLM(...)` constructs without error with **default compilation on** (no `enforce_eager`).
2. `collective_rpc("tl_install_capture", ...)` returns success on every worker.
3. `llm.generate(...)` runs without error.
4. **Hook fire count over 100 runs == 100** (the load-bearing test: hooks must survive `torch.compile` + CUDA-graph replay).
5. All 100 captures **bitwise identical** (no race conditions / stream-sync bugs).
6. vLLM captured tensor and HF reference tensor norms within ~0.5–2.0× of each other (sanity that the hook caught real attention output, not noise).

In [ ]:
!pip install -q vllm transformers
import vllm, torch
print("vllm:", vllm.__version__)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

In [ ]:
# Optional HF auth — only needed for gated models (Llama-3.2-1B etc.).
# The default Qwen2.5-0.5B-Instruct does not require auth; this block is a no-op for it.
import os
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        from huggingface_hub import login
        login(token=tok, add_to_git_credential=False)
        print("HF auth OK")
    else:
        print("No HF_TOKEN secret set; that's fine for non-gated models.")
except Exception as e:
    print(f"Skipping HF auth ({e}); fine for non-gated models.")

In [ ]:
%%writefile /content/tl_spike_ext.py
"""Phase 0b spike — minimal vLLM WorkerExtension.

Registers a single PyTorch forward hook on the module at a dot-path inside
the worker's loaded model, stashes outputs into a per-worker buffer, and
exposes read/reset methods to the driver via collective_rpc.

The hook is installed BEFORE any generate() call — this is the central claim
the spike validates: hooks present at compile time survive torch.compile and
CUDA-graph capture.
"""
from typing import List
import torch


class TLSpikeExtension:
    """Mixed into vLLM's Worker. Methods exposed via collective_rpc."""

    def tl_install_capture(self, dot_path: str) -> str:
        model = self.model_runner.model
        target = model
        for seg in dot_path.split("."):
            target = target[int(seg)] if seg.isdigit() else getattr(target, seg)

        if not hasattr(self, "_tl_captures"):
            self._tl_captures: List[torch.Tensor] = []
            self._tl_fire_count: int = 0

        @torch.no_grad()
        def _hook(_module, _inputs, output):
            tensor = output[0] if isinstance(output, tuple) else output
            if isinstance(tensor, torch.Tensor):
                torch.cuda.current_stream().synchronize()
                self._tl_captures.append(tensor.detach().clone().cpu())
                self._tl_fire_count += 1

        target.register_forward_hook(_hook)
        return f"hook installed on {dot_path}; module: {type(target).__name__}"

    def tl_read_captures(self) -> List[torch.Tensor]:
        out = getattr(self, "_tl_captures", [])
        self._tl_captures = []
        return out

    def tl_read_fire_count(self) -> int:
        return getattr(self, "_tl_fire_count", 0)

    def tl_reset(self) -> None:
        self._tl_captures = []
        self._tl_fire_count = 0

In [ ]:
# vLLM uses `spawn` (not fork) for its worker subprocess once CUDA is initialized in
# the parent — see the earlier WARNING about VLLM_WORKER_MULTIPROC_METHOD. Spawned
# workers do NOT inherit sys.path from the parent kernel. They DO inherit os.environ.
# So we set PYTHONPATH (which propagates) and also copy the extension into site-packages
# as a belt-and-suspenders. Also turn on DEBUG logging so any next failure surfaces a
# real traceback instead of the generic 'Engine core initialization failed'.
import os, sys, shutil, site
os.environ["PYTHONPATH"] = "/content" + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
sys.path.insert(0, "/content")
_site = site.getsitepackages()[0]
shutil.copy("/content/tl_spike_ext.py", _site)
print(f"PYTHONPATH: {os.environ['PYTHONPATH']}")
print(f"Copied tl_spike_ext.py → {_site}")

# Verify the worker WILL be able to import the extension by importing it here too.
if "tl_spike_ext" in sys.modules:
    del sys.modules["tl_spike_ext"]
import tl_spike_ext
assert hasattr(tl_spike_ext, "TLSpikeExtension"), "TLSpikeExtension not found in tl_spike_ext module"
print(f"Verified: tl_spike_ext.TLSpikeExtension importable from {tl_spike_ext.__file__}")

from vllm import LLM, SamplingParams
from vllm.inputs import TokensPrompt

# Default: non-gated, ~1GB. Swap to 'meta-llama/Llama-3.2-1B' if you have HF access.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Dot-path of the layer-0 self-attention module, relative to model_runner.model.
# For both Qwen2 and Llama vLLM models, the path is `model.layers.0.self_attn`.
TARGET_DOT_PATH = "model.layers.0.self_attn"

print(f"\nConstructing LLM for {MODEL_NAME} (compilation ON — this takes a few minutes)...")
llm = LLM(
    model=MODEL_NAME,
    tensor_parallel_size=1,
    skip_tokenizer_init=True,
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
    dtype="bfloat16",
    worker_extension_cls="tl_spike_ext.TLSpikeExtension",
    # Notably absent: no enforce_eager, no compilation_config override, no max_num_seqs override.
    # vLLM's full V1 engine runs — torch.compile + piecewise CUDA graphs all on.
)
print("LLM constructed.")

In [ ]:
# Install the capture hook BEFORE the first generate() call.
# This is the load-bearing decision: hooks must be present at compile time.
install_result = llm.collective_rpc("tl_install_capture", args=(TARGET_DOT_PATH,))
print("install_capture (one entry per worker):", install_result)

# Tokenize prompt with HF tokenizer (vLLM's was skipped).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
PROMPT = "The quick brown fox jumps over the"
ids = tok(PROMPT, add_special_tokens=True).input_ids
print(f"Prompt token count: {len(ids)}")

# Run a single generate(). vLLM will compile + capture CUDA graphs on this call.
out = llm.generate(
    prompts=[TokensPrompt(prompt_token_ids=ids)],
    sampling_params=SamplingParams(max_tokens=1, temperature=0.0),
)
print("generated token:", out[0].outputs[0].token_ids)

caps_per_worker = llm.collective_rpc("tl_read_captures")
fire_per_worker = llm.collective_rpc("tl_read_fire_count")
print(f"\nHook fire count per worker after 1 generate: {fire_per_worker}")
print(f"Capture list length per worker: {[len(c) for c in caps_per_worker]}")
if caps_per_worker and caps_per_worker[0]:
    print(f"First capture shape: {caps_per_worker[0][0].shape}, dtype: {caps_per_worker[0][0].dtype}")
    print(f"First capture norm: {caps_per_worker[0][0].float().norm().item():.4f}")

In [ ]:
# Determinism check — 100 back-to-back generates with the same prompt.
# Pass criteria: fire_count == 100 AND all 100 captures bitwise identical.
import time

llm.collective_rpc("tl_reset")
t0 = time.time()
for _ in range(100):
    llm.generate(
        prompts=[TokensPrompt(prompt_token_ids=ids)],
        sampling_params=SamplingParams(max_tokens=1, temperature=0.0),
    )
elapsed = time.time() - t0

fire_count = llm.collective_rpc("tl_read_fire_count")[0]
caps = llm.collective_rpc("tl_read_captures")[0]

print(f"100 generations in {elapsed:.1f}s  ({elapsed*10:.0f} ms/gen)")
print(f"Hook fire count: {fire_count}  (expected: 100)")
print(f"Captures collected: {len(caps)}")

if len(caps) >= 2:
    ref = caps[0]
    all_equal = all(torch.equal(ref, c) for c in caps[1:])
    print(f"All captures bitwise identical: {all_equal}")
    if not all_equal:
        diffs = [int((ref != c).any().item()) for c in caps[1:]]
        print(f"Non-matching capture count: {sum(diffs)} / {len(caps) - 1}")
        first_bad = next((i+1 for i, d in enumerate(diffs) if d), None)
        if first_bad is not None:
            delta = (ref - caps[first_bad]).float().abs().max().item()
            print(f"First mismatching capture at idx {first_bad}; max abs delta: {delta:.6f}")

In [ ]:
# HF reference forward — sanity check that the hook caught the real attention output,
# not an empty buffer or constant. Shapes will differ (vLLM fuses QKV, HF does not),
# but norms should be within a small constant factor.
import torch
from transformers import AutoModelForCausalLM

print(f"Loading HF reference model {MODEL_NAME}...")
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda"
).eval()

hf_caps = []
def hf_hook(_m, _i, output):
    t = output[0] if isinstance(output, tuple) else output
    if isinstance(t, torch.Tensor):
        hf_caps.append(t.detach().clone().cpu())

target = hf_model
for seg in TARGET_DOT_PATH.split("."):
    target = target[int(seg)] if seg.isdigit() else getattr(target, seg)
handle = target.register_forward_hook(hf_hook)

with torch.no_grad():
    hf_model(torch.tensor([ids], device="cuda"))
handle.remove()

hf_tensor = hf_caps[0]
vllm_tensor = caps[0]
print(f"\nHF   tensor: shape={tuple(hf_tensor.shape)}  norm={hf_tensor.float().norm().item():.4f}")
print(f"vLLM tensor: shape={tuple(vllm_tensor.shape)}  norm={vllm_tensor.float().norm().item():.4f}")

hf_norm   = hf_tensor.float().norm().item()
vllm_norm = vllm_tensor.float().norm().item()
ratio = vllm_norm / hf_norm if hf_norm > 0 else float("inf")
print(f"Norm ratio (vLLM / HF): {ratio:.3f}  — expect ~0.5–2.0; orders-of-magnitude difference = bug")

In [ ]:
# Phase 0b Summary — paste this entire output back to Claude
import vllm, torch
print("=" * 60)
print("PHASE 0b SUMMARY")
print("=" * 60)
print(f"vllm:           {vllm.__version__}")
print(f"torch:          {torch.__version__}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"Model:          {MODEL_NAME}")
print(f"Target path:    {TARGET_DOT_PATH}")
print()
print(f"Construct OK:   {llm is not None}")
print(f"Install OK:     {bool(install_result and install_result[0])}")
print(f"Fire count:     {fire_count} / 100   {'PASS' if fire_count == 100 else 'FAIL'}")
if len(caps) >= 2:
    all_equal = all(torch.equal(caps[0], c) for c in caps[1:])
    print(f"Bitwise det.:   {all_equal}    {'PASS' if all_equal else 'FAIL'}")
print(f"Norm ratio:     {ratio:.3f}  {'PASS' if 0.5 <= ratio <= 2.0 else 'INVESTIGATE'}")
print(f"Latency:        {elapsed*10:.0f} ms/gen ({elapsed:.1f}s for 100 runs)")
print("=" * 60)

## Interpreting results

| Outcome | What it means |
|---|---|
| All 5 checks PASS | **Go.** Plugin path is the correct architecture for v1. Proceed to Phase 0a + 1. |
| Construct fails on `worker_extension_cls` kwarg | API name may have changed in your vLLM version. Check `vllm.LLM.__init__` signature; common alternatives: `worker_cls`, `additional_config={"worker_extension_cls": ...}`. |
| Construct OK but fire_count == 0 | Hook is installed but never fires — most likely `torch.compile` stripped it. **No-go on current vLLM.** Wait for [RFC #36998](https://github.com/vllm-project/vllm/issues/36998). |
| fire_count > 0 but < 100 | Hook fires intermittently — race or compile boundary issue. Diagnose before committing to v1. |
| fire_count == 100 but captures NOT bitwise identical | Missing stream sync inside hook, or KV cache pollution. The `synchronize()` in the hook should prevent this; if it still happens, deeper investigation needed. |
| Norm ratio wildly off | Hook is firing but on wrong tensor / wrong shape. Check `TARGET_DOT_PATH` is the right module for the chosen model family. |